In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

import yaml

# %matplotlib tk

In [ ]:
energy_df = pd.read_excel('Dataset.xlsx', sheet_name='2023 data', usecols='A:F', nrows=8762)
meta_data = yaml.safe_load(open("meta_data.yml", "r"))
energy_df.tail()

## Simulated Real-Time prices

In [ ]:
rt_prices_df = pd.read_csv('generated/simulated_rt_prices_year.csv')
rt_prices_df["Date"] = pd.to_datetime(rt_prices_df["Date"])
rt_prices_df.head()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(energy_df['Energy price (EUR/kWh)'], label='Day-Ahead Price')
plt.plot(rt_prices_df['RT energy price (EUR/kWh)'], label='Simulated Real-Time Price')
plt.title(f"Energy Prices")
plt.xlabel('Time')
plt.ylabel('Price (EUR/kWh)')
plt.legend()
plt.tight_layout()

In [ ]:
date = "2023-08-08"
day_rt_prices = rt_prices_df[rt_prices_df['Date'].dt.date == pd.to_datetime(date).date()]
day_da_prices = energy_df[energy_df['Date'].dt.date == pd.to_datetime(date).date()]

plt.figure(figsize=(12, 6))
plt.plot(day_da_prices['Energy price (EUR/kWh)'], label='Day-Ahead Price')
plt.plot(day_rt_prices['RT energy price (EUR/kWh)'], label='Simulated Real-Time Price')
plt.title(f"Energy Prices on {date}")
plt.xlabel("Hour")
plt.ylabel("Price (EUR/kWh)")
plt.legend()
plt.tight_layout()

## Checking the generated data

In [ ]:
perfect_foresight_df = pd.read_csv('./generated/perfect_foresight_optimization_year.csv')
perfect_foresight_df['Date'] = pd.to_datetime(perfect_foresight_df['Date'])
perfect_foresight_df.head()

In [ ]:
num_scenarios = 9

stochastic_optim_df = pd.read_csv(f'./generated/stochastic_optimization_with_{num_scenarios}_scenarios_year.csv')
stochastic_optim_df['Date'] = pd.to_datetime(stochastic_optim_df['Date'])
stochastic_optim_df.head()

In [ ]:
file_path = f'./generated/stochastic_optimization_with_{num_scenarios}_scenarios_year_zero_noise.csv'
zero_noise_data_exists = os.path.exists(file_path)

if zero_noise_data_exists:
    stochastic_optim_zero_noise_df = pd.read_csv(file_path)
    stochastic_optim_zero_noise_df['Date'] = pd.to_datetime(stochastic_optim_zero_noise_df['Date'])
    stochastic_optim_zero_noise_df.head()

In [ ]:
stochastic_policy_evaluation_df = pd.read_csv('./generated/stochastic_policy_evaluation_year.csv')
stochastic_policy_evaluation_df['Date'] = pd.to_datetime(stochastic_policy_evaluation_df['Date'])
stochastic_policy_evaluation_df.head()

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

axs[0].plot(perfect_foresight_df['grid_import'], '--', c='grey', label='Perfect Foresight - Import')
axs[0].plot(stochastic_optim_df['grid_import_ahead'], label='Stochastic Optimization - Ahead Import')
axs[0].plot(stochastic_optim_df['expected_grid_import_rt'], label='Stochastic Optimization - Expected RT Import')
axs[0].legend()
axs[0].set_title('Stochastic Optimization Grid Import Comparison')
axs[0].set_xlabel('Time')
axs[0].set_ylabel('Import (kW)')

if zero_noise_data_exists:
    axs[1].plot(perfect_foresight_df['grid_import'], '--', c='grey', label='Perfect Foresight - Import')
    axs[1].plot(stochastic_optim_zero_noise_df['grid_import_ahead'], label='Stochastic Optimization (Zero Noise) - Ahead Import')
    axs[1].plot(stochastic_optim_zero_noise_df['expected_grid_import_rt'], label='Stochastic Optimization (Zero Noise) - Expected RT Import')
    axs[1].legend()
    # axs[1].set_title('Stochastic Optimization Ahead Grid Import Comparison')
    axs[1].set_xlabel('Time')
    axs[1].set_ylabel('Import (kW)')

plt.tight_layout()

In [ ]:
stochastic_final_df = stochastic_optim_df.merge(stochastic_policy_evaluation_df, on='Date', suffixes=('_optimization', '_evaluation'), how='left')

stochastic_final_df.info()

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

axs[0].plot(perfect_foresight_df['grid_import'], '--', c='grey', label='Perfect Foresight - Import')
axs[0].plot(stochastic_final_df['grid_import_ahead_optimization'], label='Stochastic Optimization - Ahead Import')
axs[0].plot(stochastic_final_df['expected_grid_import_rt_optimization'], label='Stochastic Optimization - Expected RT Import')
axs[0].legend()
axs[0].set_title('Stochastic Policy Evaluation Grid Import Comparison')
axs[0].set_xlabel('Time')
axs[0].set_ylabel('Import (kW)')

axs[1].plot(perfect_foresight_df['grid_import'], '--', c='grey', label='Perfect Foresight - Import')
axs[1].plot(stochastic_final_df['grid_import_ahead_evaluation'], label='Stochastic Policy Evaluation - Ahead Import')
axs[1].plot(stochastic_final_df['expected_grid_import_rt_evaluation'], label='Stochastic Policy Evaluation - Expected RT Import')
axs[1].legend()
# axs[1].set_title('Stochastic Optimization Ahead Grid Import Comparison')
axs[1].set_xlabel('Time')
axs[1].set_ylabel('Import (kW)')

plt.tight_layout()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(perfect_foresight_df['price'], '--', c='grey', label='Perfect Foresight - Price')
plt.plot(stochastic_final_df['price_ahead_optimization'], label='Stochastic Optimization - Ahead Price')
plt.plot(stochastic_final_df['price_ahead_evaluation'], label='Stochastic Optimization Evaluation - Ahead Price')

plt.legend()
plt.title('Stochastic Optimization Ahead Price Comparison')
plt.xlabel('Time')
plt.ylabel('Price (EUR/kWh)')
plt.tight_layout()

non_nans = ~stochastic_final_df['price_ahead_evaluation'].isna()
print(f"Number of days evaluated: {len(non_nans) // 24}")
assert np.allclose(stochastic_final_df['price_ahead_optimization'][non_nans], stochastic_final_df['price_ahead_evaluation'][non_nans], atol=1e-3), "Prices do not match!"

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(perfect_foresight_df['grid_import'], '--', c='grey', label='Perfect Foresight - Import')
plt.plot(stochastic_final_df['grid_import_optimization'], label='Stochastic Optimization - Import')
plt.plot(stochastic_final_df['grid_import_evaluation'], label='Stochastic Evaluation - Import')
plt.legend()
plt.title('Stochastic Optimization Grid Import Comparison')
plt.xlabel('Time')
plt.ylabel('Import (kW)')

plt.tight_layout()

In [ ]:
num_batteries = sum(["battery" in col for col in perfect_foresight_df.columns])
num_batteries //= 3

fig, axs = plt.subplots(num_batteries, 1, figsize=(12, 4*num_batteries), sharex=True)

for i in range(num_batteries):
    axs[i].plot(perfect_foresight_df[f'battery_{i+1}_soc'], '--', c='grey', label=f'Perfect Foresight - Battery {i+1} SoC')

    # axs[i].plot(stochastic_final_df[f'grid_import_evaluation'], label=f'Stochastic Policy Evaluation - Grid Import')
    axs[i].plot(stochastic_final_df[f'expected_battery_{i+1}_soc_optimization'], label=f'Stochastic Optimization - Battery {i+1} SoC')
    axs[i].plot(stochastic_final_df[f'expected_battery_{i+1}_soc_evaluation'], label=f'Stochastic Policy Evaluation - Battery {i+1} SoC')

    axs[i].legend()
    axs[i].set_title(f'Battery {i+1} SoC Comparison')
    axs[i].set_xlabel('Time')
    axs[i].set_ylabel('State of Charge (kWh)/Grid Import (kW)')

In [ ]:
fig, axs = plt.subplots(1, num_batteries, figsize=(6*num_batteries, 6))

for i in range(num_batteries):
    battery_soc_x = perfect_foresight_df[f'battery_{i+1}_soc'].values.reshape(-1, 24)
    battery_soc_y_1 = stochastic_final_df[f'expected_battery_{i+1}_soc_optimization'].values.reshape(-1, 24)
    battery_soc_y_2 = stochastic_final_df[f'expected_battery_{i+1}_soc_evaluation'].values.reshape(-1, 24)
    axs[i].plot(battery_soc_x[:,0], battery_soc_y_1[:,0], 'o', label=f'Stochastic Optimization - Battery {i+1} SoC')
    axs[i].plot(battery_soc_x[:,0], battery_soc_y_2[:,0], 'o', label=f'Stochastic Policy Evaluation - Battery {i+1} SoC')

    axs[i].legend()
    axs[i].set_title(f'Battery {i+1} SoC Comparison (Daily Start)')

plt.tight_layout()


### Price Comparison

In [ ]:
# Perfect Foresight
perfect_foresight_cost = (perfect_foresight_df['price'] * perfect_foresight_df['grid_import'])[non_nans].sum()
print("Total Cost with Perfect Foresight: {:.2f} EUR".format(perfect_foresight_cost))

# Stochastic Policy Evaluation
ahead_cost_evaluation = (stochastic_final_df['price_ahead_evaluation'] * stochastic_final_df['grid_import_ahead_evaluation'])[non_nans].sum()

price_rt = rt_prices_df['RT energy price (EUR/kWh)'].values
rt_cost_evaluation = (price_rt * stochastic_final_df['expected_grid_import_rt_evaluation'])[non_nans].sum()
print("Total Cost with Stochastic Policy Evaluation: {:.2f} EUR".format(ahead_cost_evaluation + rt_cost_evaluation))

diff = (ahead_cost_evaluation + rt_cost_evaluation) - perfect_foresight_cost
print("Cost of Lack of Perfect Foresight: {:.2f} EUR ({:.2f} %)".format(diff, diff / perfect_foresight_cost * 100))